In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini", trigger=("tokens", 100), keep=("messages", 1)
        )
    ],
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="What is the capital of the moon?"),
            AIMessage(content="The capital of the moon is Lunapolis."),
            HumanMessage(content="What is the weather in Lunapolis?"),
            AIMessage(
                content="Skies are clear, with a high of 120C and a low of -100C."
            ),
            HumanMessage(content="How many cheese miners live in Lunapolis?"),
            AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
            HumanMessage(content="Do you think the cheese miners' union will strike?"),
            AIMessage(content="Yes, because they are unhappy with the new president."),
            HumanMessage(
                content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"
            ),
        ]
    },
    {"configurable": {"thread_id": "1"}},
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user seeks information about the fictional city of Lunapolis on the moon, including its capital, weather, population of cheese miners, and the status of their union.\n\n## SUMMARY\n- The capital of the moon is identified as Lunapolis.\n- The weather in Lunapolis is described as clear skies, with temperatures ranging from a high of 120°C to a low of -100°C.\n- The population of cheese miners in Lunapolis is stated to be 100,000.\n- The cheese miners' union is expected to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\nNone", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='057a770d-dc5d-404c-a21e-69cc0690eb0c'),
              HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?", additional_kwargs={}, response_metadata={}, id='7fae15c9-331e-48c1-9d72-8

In [5]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user seeks information about the fictional city of Lunapolis on the moon, including its capital, weather, population of cheese miners, and the status of their union.

## SUMMARY
- The capital of the moon is identified as Lunapolis.
- The weather in Lunapolis is described as clear skies, with temperatures ranging from a high of 120°C to a low of -100°C.
- The population of cheese miners in Lunapolis is stated to be 100,000.
- The cheese miners' union is expected to strike due to dissatisfaction with the new president.

## ARTIFACTS
None

## NEXT STEPS
None


## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage


@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]

    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [18]:
agent = create_agent(
    model="gpt-5-nano",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [ ]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="My device won't turn on. What should I do?"),
            ToolMessage(
                content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"
            ),
            AIMessage(content="Is the device plugged in and turned on?"),
            HumanMessage(content="Yes, it's plugged in and turned on."),
            ToolMessage(
                content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"
            ),
            AIMessage(content="Is the device showing any lights or indicators?"),
            HumanMessage(content="What's the temperature of the device?"),
        ]
    },
    {"configurable": {"thread_id": "2"}},
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='438de54b-7883-4564-bb5d-51f851974ff5'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='7032f4d1-9d55-48a5-87bf-7e82a2dd3463', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='ef18ddc0-fcc2-4402-8704-eeab87b2da3a'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='3d285781-dac7-4c5a-8207-da99cc276ec4', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='4093d432-4350-449d-b433-2a1a2c9d6e66'),
              AIMessage(content='I can’t read the device’s temperature from here. If the device

In [20]:
print(response["messages"][-1].content)

I can’t read the device’s temperature from here. If the device isn’t turning on, there won’t be a temperature reading. If it does power on, you can check temperature locally with one of these:

- PC or laptop (Windows/macOS/Linux)
  - BIOS/UEFI: Restart and enter BIOS/UEFI (keys vary, e.g., F2, Del, F10). Look for a Hardware Monitor or System Health section with CPU/GPU temps.
  - In the OS: use a third‑party tool such as HWMonitor, Open Hardware Monitor, or Core Temp (Windows) or iStat Menus / Macs Fan Control (macOS).

- Desktop or laptop with Windows: you can also view some sensor data in HWMonitor/Open Hardware Monitor after installing and running them.

- Smartphones/Tablets
  - Android: apps like CPU-Z or AIDA64 can show temperatures.
  - iPhone/iPad: iOS doesn’t expose temps in Settings; third‑party apps may show limited data, but hardware temps aren’t always accessible.

If you want, tell me what device you’re using (brand/model and OS), and I’ll give you exact steps and recomm